# CourtVisionNet Training on Kaggle

Trains CourtVisionNet on the leak-free, group-split dataset and evaluates it on the
held-out `test_real` and `test_roboflow` splits.

**Prerequisites:**
1. Build the dataset locally: `python scripts/prepare_colab.py --zip` (writes `data/colab_data_v2.zip`)
2. Upload `colab_data_v2.zip` (or the unzipped `colab_data_v2/` folder) as a Kaggle Dataset and add it to this notebook
3. Settings → Accelerator → **GPU**; Settings → Internet → **On**
4. Run all cells in order

## 1. Setup

In [ ]:
# === CONFIGURE THESE ===
KAGGLE_DATASET = 'cvn-colab-data-v2'   # your Kaggle dataset name
REPO_URL = 'https://github.com/roasteduck04/badminton-court-finder.git'
REPO_REF = 'main'                      # pin a commit SHA for reported runs
CONFIG = 'configs/baseline.json'
RUN_NAME = 'baseline'
SEEDS = [0, 1, 2]
EXTRA_OVERRIDES = []                   # e.g. ['num_epochs=50', 'batch_size=16']
RESUME_FROM = ''                       # e.g. '/kaggle/input/<prev-output>/runs/baseline_s0/last.pt'
NUM_WORKERS = 4

import os
KAGGLE_INPUT = f'/kaggle/input/{KAGGLE_DATASET}'
WORK = '/kaggle/working'
REPO_DIR = f'{WORK}/badminton-court-finder'
OUT_DIR = f'{WORK}/runs'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Dataset path: {KAGGLE_INPUT}: {os.listdir(KAGGLE_INPUT)}')

- `REPO_REF`: pin to a commit SHA for any run you report (the SHA is also stored in every checkpoint).
- `CONFIG`: `configs/baseline.json` or an ablation from `configs/` (see `configs/README.md`).
- `SEEDS`: use at least 3 seeds per config for reported numbers.

In [ ]:
# Clone the repo and check out the pinned ref (commit SHA, tag or branch)
import os
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch -q --all --tags
!git -C {REPO_DIR} checkout -q {REPO_REF}
!git -C {REPO_DIR} log -1 --format='Using commit %H (%s)'
%cd {REPO_DIR}

In [ ]:
# torch/torchvision are preinstalled; the code needs the albumentations 2.x API
!pip install -q 'albumentations>=2.0.8,<3' 'shapely>=2.0,<3'

In [ ]:
# Locate the dataset: zip upload or folder upload
import zipfile

DATA_DIR = None
for candidate in [f'{WORK}/colab_data_v2', f'{KAGGLE_INPUT}/colab_data_v2', KAGGLE_INPUT]:
    if os.path.isfile(os.path.join(candidate, 'manifest.json')):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    zip_path = os.path.join(KAGGLE_INPUT, 'colab_data_v2.zip')
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f'No manifest.json or colab_data_v2.zip in {KAGGLE_INPUT}')
    print(f'Unzipping {zip_path} ...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK)
    DATA_DIR = f'{WORK}/colab_data_v2'
print(f'Using data from {DATA_DIR}')

In [ ]:
# Verify the manifest and its splits
import json
MANIFEST = os.path.join(DATA_DIR, 'manifest.json')
with open(MANIFEST, encoding='utf-8') as f:
    manifest = json.load(f)
for split, entries in manifest['splits'].items():
    by_source = {}
    for e in entries:
        by_source[e['source']] = by_source.get(e['source'], 0) + 1
    groups = len({e['group'] for e in entries})
    print(f'  {split:<14} {len(entries):5d} images, {groups:3d} groups  {by_source}')

## 2. Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 3. Train

In [ ]:
# Train one run per seed. Each run writes best_model.pt, last.pt,
# config.json and history.json to {OUT_DIR}/{name}_s{seed}.
def run_dir(seed):
    return f'{OUT_DIR}/{RUN_NAME}_s{seed}'

for seed in SEEDS:
    overrides = [f'seed={seed}', f'manifest={MANIFEST}', f'name={RUN_NAME}',
                 f'checkpoint_dir={run_dir(seed)}', f'num_workers={NUM_WORKERS}'] + EXTRA_OVERRIDES
    resume = f'--resume-from {RESUME_FROM}' if RESUME_FROM else ''
    print(f'===== {CONFIG} seed={seed} =====')
    !python -m src.training.train --config {CONFIG} {resume} --set {' '.join(overrides)}

## 4. Evaluate on held-out test splits

In [ ]:
# Evaluate each run's best checkpoint on the two held-out test splits
# separately (never on val, which was used for model selection).
for seed in SEEDS:
    ckpt = f'{run_dir(seed)}/best_model.pt'
    for split in ['test_real', 'test_roboflow']:
        print(f'===== seed={seed} {split} =====')
        !python -m src.evaluation.evaluate --checkpoint {ckpt} --manifest {MANIFEST} --split {split} --out {run_dir(seed)}/eval_{split}.json

In [ ]:
# Headline test numbers per seed (full tables are in eval_*.md next to each checkpoint)
import glob
for path in sorted(glob.glob(f'{OUT_DIR}/{RUN_NAME}_s*/eval_*.json')):
    with open(path, encoding='utf-8') as f:
        summary = json.load(f)
    print(path)
    print(json.dumps(summary.get('headline', summary), indent=1)[:2000])

## 5. Training history

In [ ]:
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for seed in SEEDS:
    path = f'{run_dir(seed)}/history.json'
    if not os.path.exists(path):
        continue
    with open(path, encoding='utf-8') as f:
        history = json.load(f)
    ep = [h['epoch'] for h in history]
    axes[0].plot(ep, [h['train_loss'] for h in history], label=f'train s{seed}')
    axes[0].plot(ep, [h['val_loss'] for h in history], '--', label=f'val s{seed}')
    axes[1].plot(ep, [h['val_pck10'] for h in history], label=f'PCK@10 s{seed}')
    axes[1].plot(ep, [h['val_pck5'] for h in history], ':', label=f'PCK@5 s{seed}')
    axes[2].plot(ep, [h['val_mean_px'] for h in history], label=f's{seed}')
axes[0].set_title('Loss'); axes[1].set_title('Val PCK (640 canvas px)'); axes[2].set_title('Val mean error (px)')
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_history.png', dpi=150)
plt.show()

## 6. Outputs

Everything under `/kaggle/working/runs/` (checkpoints, `history.json`, `config.json`,
evaluation summaries) is kept as notebook output. Click **Save Version** → **Save & Run All**,
then download from the notebook's **Output** tab. To resume a run, add the previous output as
a dataset and set `RESUME_FROM` to its `last.pt`.